# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os

REPO_URL = "https://github.com/tkg-create/FlyRank-ML-Track.git"
REPO_DIR = "FlyRank-ML-Track"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
assert os.path.isfile("work/scripts/01_load_and_score.py"), "01_load_and_score.py not found — did the clone work?"

Cloning into 'FlyRank-ML-Track'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (266/266), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 266 (delta 131), reused 114 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (266/266), 1.96 MiB | 11.90 MiB/s, done.
Resolving deltas: 100% (131/131), done.
Working dir: /content/FlyRank-ML-Track


## 1. Question

*The research question and the decision it supports.*

Organizations that manage content across many sites, such as SEO teams, agencies, and large publishers, share a common problem. The portfolio of pages they manage is far larger than their capacity to review it by hand. Editorial attention is scarce, so it has to be spent on the pages where it matters most. Given a fixed review capacity, which declining pages should be reviewed first, and what should a reviewer actually do with each one?

This work answers that question for a portfolio of thousands of client sites and millions of pages. Review capacity is set at 50 pages per month (`REVIEW_CAPACITY_K`, defined below). That number is an assumed budget rather than a given, chosen because it's also the point where a model-informed ranking held its advantage over a hand-built rule most consistently under grouped cross-validation, winning in 5 of 5 folds (see Methodology and Results), rather than a K that simply looked best on average.

In [5]:
"""
Assumed monthly review capacity — chosen because grouped cross-validation found
this is where a model-informed ranking's edge over the rule is most consistent (5/5 folds),
not just where it looks best on average. See Methodology and Results.
"""
REVIEW_CAPACITY_K = 50

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Tables:** Each row represents one page, on one date, for one client. The table is `fact_content_daily_performance`, filtered to the `month=2026-03` partition of the internship warehouse. That month was chosen deliberately. Earlier months are missing analytics history for many clients, and later months overlap a separate 90-day table's window in a way that risks the label and features bleeding into each other. A month in the middle of the panel avoids both problems.

**Date windows:** The single month is split into two windows. The first half of the month versus the second half, March 1 to 15 against March 16 to 31, defines the decline label. Week 1 versus week 2 of the first half, March 1 to 8 against March 9 to 15, defines the position-trend feature. The feature window sits strictly before the label window, so no feature can see into the period it's predicting.

**Excluded, and why:** The raw first-half and second-half impression counts that the label is built from are excluded from every feature set. Adding one of them back in as a diagnostic pushed a model's ROC AUC from 0.564 to 0.991, which meant the model was reconstructing the label algebraically rather than learning anything real. A content-metadata table was excluded because it turned out to be a current-state snapshot rather than a point-in-time history. Joining it onto daily rows produced impossible negative day counts. Rows without a confirmed analytics coverage flag were excluded, since those rows measure "not tracked" rather than a genuine zero. A session-rate engagement signal was excluded as well, after earlier analysis found it confounded and pointing in the wrong direction, and has been left out of every model built since.

**Public-safe:** Every identifier used for joining or grouping is a hashed ID and is never used as a model feature. No client name, URL, or search query text appears anywhere in this pipeline's outputs.

In [6]:
import subprocess
import sys
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face READ token: ")

process = subprocess.Popen(
    [sys.executable, "-u", "work/scripts/01_load_and_score.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
process.wait()
assert process.returncode == 0, f"01_load_and_score.py failed with exit code {process.returncode}"

Paste your Hugging Face READ token: ··········
Loading and building features from the warehouse...
  [1/6] Querying label (impressions first half vs second half)...
        -> 151,981 rows
  [2/6] Querying full-month position/click signal (baseline rule)...
        -> 175,304 rows
  [3/6] Querying first-half-only features (model training data)...
        -> 150,675 rows
  [4/6] Querying leakage-safe position trend (week 1 vs week 2)...
        -> 119,176 rows
  [5/6] Querying client map (grouping key only, never a feature)...
        -> 331,437 rows
  [6/6] Merging into model_df...
model_df: (150675, 16), base rate: 0.438
Running 5-fold GroupKFold OOF scoring (random_state=42)...
  Fold 1/5: fitting on 120,583 rows, scoring 30,092...
  Fold 1/5: done
  Fold 2/5: fitting on 120,589 rows, scoring 30,086...
  Fold 2/5: done
  Fold 3/5: fitting on 120,351 rows, scoring 30,324...
  Fold 3/5: done
  Fold 4/5: fitting on 120,589 rows, scoring 30,086...
  Fold 4/5: done
  Fold 5/5: fitting on 

In [7]:
import pandas as pd

model_df = pd.read_csv("work/data/processed/w07_scored_population.csv")
print(f"Rows: {len(model_df):,}")
print(f"Base rate (share labeled declining): {model_df['is_declining_proxy'].mean():.3f}")
model_df[["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh", "position_change", "has_position_trend"]].describe()

Rows: 150,675
Base rate (share labeled declining): 0.438


,avg_position_fh,log_impressions_fh,log_clicks_fh,ctr_fh,position_change,has_position_trend
count,150675.000000,150675.000000,150675.000000,150675.000000,150675.000000,150675.000000
mean,16.546579,4.630053,0.526925,0.004285,0.618596,0.790947
std,18.215346,2.287959,0.901977,0.034230,9.208292,0.406633
min,0.040763,0.693147,0.000000,0.000000,-213.750000,0.000000
25%,5.186723,2.833213,0.000000,0.000000,-0.950392,1.000000
50%,8.756892,4.709530,0.000000,0.000000,0.000000,1.000000
75%,21.131311,6.410175,0.693147,0.002023,1.666203,1.000000
max,310.000000,11.992731,7.781556,1.000000,153.750000,1.000000


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:** Pages need at least 10 impressions across the month to be trusted for position and click-through numbers. Pages below that bar are excluded rather than treated as having a real value of zero. The decline label is a proxy, not proof of real decline. Comparing impressions across two halves of one month captures a within-month direction, not a causal or long-term trend. Validation is grouped by client, on the assumption that a model shouldn't be tested on a client whose other pages it trained on. That's a reasonable default, but not the only possible choice; grouping by content topic or launch date was not tested.

**Model and features:** The model is a Random Forest with 300 trees, a max depth of 6, and a minimum of 20 samples per leaf. It trains on six features: average position, log impressions, log clicks, click-through rate, position change, and whether a position trend exists for the page at all. A seventh candidate feature was built during earlier validation work, a zero-click flag similar to the one used in the baseline rule. It was left out of the final feature set on purpose, since it only ever backed an earlier diagnostic comparison. As a result the model sees click-through problems only indirectly, through CTR and click volume, rather than as an explicit flag the way the baseline rule does.

**Label:** A page is labeled declining by comparing its search impressions in the first half of the month to the second half. The raw impression counts from either half are never allowed as features. This was confirmed necessary rather than just cautious. Adding one of those counts back in as a diagnostic pushed a model from barely above chance to near perfect. That jump wasn't real learning. It was the model reconstructing the label algebraically from a column that was almost the label itself.

**Baseline:** The baseline is a transparent rule with two parts. One flag catches a page holding a top-10 search position with zero clicks. The other catches a page whose position got worse from the first week of the month to the second. The zero-click flag counts for more than the position flag when they're combined into a single score. Every input is knowable at the decision moment, so nothing here requires a trained model.

**Validation design:** Validation uses 5-fold cross-validation grouped by client rather than by individual page. A model trained on some of a client's pages and tested on others would leak that client's own patterns across the split. Every score used for evaluation is out of fold, meaning no page is ever scored by a model that trained on it or on another page from the same client.

**Leakage checks:** Four checks were run before any result was trusted. The first confirmed that no feature's time window crosses into the label's window. The second confirmed that the eligibility filter used to build the label doesn't itself encode the outcome. The third confirmed that no feature is a near-duplicate of the label under a different name. The fourth was a deliberate injection check, where a known-leaky column was added on purpose to confirm the validation setup would actually catch it rather than assuming a clean-looking score always means a clean pipeline. All four passed.

In [12]:
import sys
sys.path.insert(0, "work/scripts")
from w07_pipeline_utils import FEATURE_COLS, RANDOM_STATE, N_FOLDS

print("Features:", FEATURE_COLS)
print(f"GroupKFold folds: {N_FOLDS}, grouped by client_hash_id")
print(f"Random Forest: n_estimators=300, max_depth=6, min_samples_leaf=20, random_state={RANDOM_STATE}")
print(f"Baseline rule: baseline_score = zero_clicks_at_position*2 + position_worsened")

Features: ['avg_position_fh', 'log_impressions_fh', 'log_clicks_fh', 'ctr_fh', 'position_change', 'has_position_trend']
GroupKFold folds: 5, grouped by client_hash_id
Random Forest: n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42
Baseline rule: baseline_score = zero_clicks_at_position*2 + position_worsened


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
